# SmartSort AI: perception, decision, and navigation
An independent Python implementation of the **actual TypeScript application model** for AI Fundamentals. No backend, external AI service, NumPy, or plotting package is needed. Run all cells from top to bottom. Python 3.10+ is recommended; verified with Python 3.13.2.

The robot carries one selected object. Plastic bottle is Plastic; Paper sheet and Cardboard box are Paper. The sensor is imperfect. Ground truth drives sensing and final evaluation, **never the decision planner**. Conditional independence of readings given the fixed material is an assumption.

Corresponding sources: `src/algorithms/bayes.ts`, `finiteMdp.ts`, `bfs.ts`, and `src/simulation/{controller,sensor,parameters}.ts`. `mdp.ts` is the immediate-reward baseline, not the live sequential planner. Reproduction inputs and TypeScript-generated outputs are in `reference_results.json` next to this notebook.

In [1]:
from collections import deque
from functools import lru_cache
from math import isclose, isfinite
from pathlib import Path
import json

MATERIALS = ('PLASTIC', 'PAPER')
PRIORS = {'PLASTIC': 0.5, 'PAPER': 0.5}
OBJECTS = {'plastic-bottle': 'PLASTIC', 'paper-sheet': 'PAPER', 'cardboard-box': 'PAPER'}
BINS = {'PLASTIC': 3, 'PAPER': 7}
DEFAULT_PARAMETERS = {'sensorAccuracy': 0.85, 'incorrectSortPenalty': 8, 'maxInspections': 3}

def valid_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and isfinite(value)

def validate_belief(belief):
    if not isinstance(belief, dict) or any(not valid_number(belief.get(m)) or not 0 <= belief[m] <= 1 for m in MATERIALS):
        raise ValueError('Belief must contain finite Plastic/Paper probabilities.')
    if abs(sum(belief[m] for m in MATERIALS) - 1) > 1e-10:
        raise ValueError('Belief must sum to one.')

def validate_model(model):
    for material in MATERIALS:
        values = [model.get(o, {}).get(material) for o in MATERIALS]
        if any(not valid_number(v) or not 0 <= v <= 1 for v in values) or abs(sum(values) - 1) > 1e-10:
            raise ValueError('Normalize observations for each material (columns, not rows).')

def sensor_model(parameters):
    a, penalty, limit = (parameters[k] for k in ('sensorAccuracy', 'incorrectSortPenalty', 'maxInspections'))
    if not valid_number(a) or not 0.5 <= a <= 0.95 or not valid_number(penalty) or not 0 <= penalty <= 40 or type(limit) is not int or not 1 <= limit <= 10:
        raise ValueError('Invalid application parameters.')
    plastic, paper = round(a + 0.05, 12), round(a - 0.05, 12)
    return {'PLASTIC': {'PLASTIC': plastic, 'PAPER': round(1-paper, 12)},
            'PAPER': {'PLASTIC': round(1-plastic, 12), 'PAPER': paper}}

MODEL = sensor_model(DEFAULT_PARAMETERS)
print('Prior:', PRIORS)
print('P(observation | material):', MODEL)

Prior: {'PLASTIC': 0.5, 'PAPER': 0.5}
P(observation | material): {'PLASTIC': {'PLASTIC': 0.9, 'PAPER': 0.2}, 'PAPER': {'PLASTIC': 0.1, 'PAPER': 0.8}}


## 1. Bayesian inference
For material $m$ and observation $o$:
$$P(o)=\sum_m P(o\mid m)P(m),\qquad P(m\mid o)=\frac{P(o\mid m)P(m)}{P(o)}.$$
Default likelihoods are PLASTIC: $(0.9,0.2)$ and PAPER: $(0.1,0.8)$ for true (Plastic, Paper). The columns sum to one. Evidence normalizes the weighted prior. For repeated observations, use the **previous posterior as the next prior**. A zero-evidence observation is impossible under the model and is rejected. Equal posterior probabilities deterministically propose Plastic.

In [2]:
def bayes(observation, prior=None, model=None):
    prior = dict(PRIORS if prior is None else prior)
    model = MODEL if model is None else model
    validate_belief(prior)
    validate_model(model)
    if observation not in MATERIALS:
        raise ValueError('Observation must be PLASTIC or PAPER.')
    likelihood = dict(model[observation])
    weights = {m: prior[m] * likelihood[m] for m in MATERIALS}
    evidence = sum(weights.values())
    if evidence <= 0:
        raise ValueError('Zero evidence.')
    posterior = {m: weights[m] / evidence for m in MATERIALS}
    material = 'PLASTIC' if posterior['PLASTIC'] >= posterior['PAPER'] else 'PAPER'
    return {'observation': observation, 'priors': prior, 'likelihoods': likelihood,
            'evidence': evidence, 'posteriors': posterior,
            'mostLikelyMaterial': material, 'confidence': posterior[material]}

first = bayes('PLASTIC')
print('Weights:', {m: first['priors'][m]*first['likelihoods'][m] for m in MATERIALS})
print('Evidence:', first['evidence'])
print('Posterior:', first['posteriors'])
second = bayes('PAPER', first['posteriors'])
print('After PLASTIC then PAPER:', second['posteriors'])
assert isclose(first['posteriors']['PLASTIC'], 9/11)
assert isclose(bayes('PAPER')['posteriors']['PAPER'], 8/9)
assert isclose(second['posteriors']['PAPER'], 0.64)

Weights: {'PLASTIC': 0.45, 'PAPER': 0.1}
Evidence: 0.55
Posterior: {'PLASTIC': 0.8181818181818181, 'PAPER': 0.18181818181818182}
After PLASTIC then PAPER: {'PLASTIC': 0.36, 'PAPER': 0.64}


## 2. Finite-horizon belief-state MDP
The hidden state is the fixed true material. Relative to the current maximum-posterior proposal, the outcomes are **correct** and **wrong**. The fully observed planning state is $(b,h)$: belief $b$ plus the number of **additional** inspections $h$. This is a belief-state MDP representation of a small POMDP. Belief and budget summarize the relevant past; transitions depend only on that state and the chosen action.

| Action | Correct | Wrong | Transition |
|---|---:|---:|---|
| SORT | +10 | -8 by default | Terminal |
| INSPECT AGAIN | +3 | +2 | Bayesian update after a possible observation; $h\gets h-1$ |

Let $c=\max(b)$ and $w=1-c$ (the code uses the other stored probability):
$$R_S(b)=10c-8w,\quad R_I(b)=3c+2w.$$
For inspection, $P(o\mid b)=\sum_m P(o\mid m)b(m)$ and the next belief is $B(b,o)$.
With discount factor 1:
$$V_0(b)=\max(0,R_S(b)),$$
$$Q_h(b,S)=R_S(b),\quad Q_h(b,I)=R_I(b)+\sum_oP(o\mid b)V_{h-1}(B(b,o)),$$
$$V_h(b)=\max(Q_h(b,S),Q_h(b,I))\quad(h>0).$$
At zero budget, inspection is **unavailable**, not a positive-reward action still available forever. A non-profitable SORT ends unresolved with value zero. Ties within $10^{-12}$ prefer inspection when available, otherwise unresolved. Positive inspection rewards can encourage spending the whole budget. We retain this specified reward model rather than silently adding an inspection cost. The planner sees no ground truth or future scripted/seeded readings.

In [3]:
def solve_mdp(belief, remaining, model=None, wrong_reward=-8):
    model = MODEL if model is None else model
    validate_belief(belief)
    validate_model(model)
    if type(remaining) is not int or not 0 <= remaining <= 10:
        raise ValueError('Remaining horizon must be 0..10.')
    if not valid_number(wrong_reward) or wrong_reward > 0:
        raise ValueError('Wrong-sort reward must be finite and nonpositive.')

    @lru_cache(maxsize=None)
    def solve(plastic, paper, h):
        b = {'PLASTIC': plastic, 'PAPER': paper}
        proposed = 'PLASTIC' if plastic >= paper else 'PAPER'
        other = 'PAPER' if proposed == 'PLASTIC' else 'PLASTIC'
        correct, wrong = b[proposed], b[other]
        rewards = {'SORT': 10*correct + wrong_reward*wrong,
                   'INSPECT AGAIN': 3*correct + 2*wrong}
        transitions = []
        if h > 0:
            for observation in MATERIALS:
                probability = sum(model[observation][m]*b[m] for m in MATERIALS)
                if probability == 0:
                    continue
                posterior = bayes(observation, b, model)['posteriors']
                next_value = solve(posterior['PLASTIC'], posterior['PAPER'], h-1)['value']
                transitions.append({'observation': observation, 'probability': probability,
                    'posterior': posterior, 'remainingInspections': h-1,
                    'nextValue': next_value, 'weightedValue': probability*next_value})
        inspect = rewards['INSPECT AGAIN'] + sum(t['weightedValue'] for t in transitions) if h else None
        alternative = inspect if inspect is not None else 0
        action = 'SORT' if rewards['SORT'] > alternative + 1e-12 else ('INSPECT AGAIN' if h else 'UNRESOLVED')
        value = rewards['SORT'] if action == 'SORT' else inspect if action == 'INSPECT AGAIN' else 0
        return {'horizon': h, 'proposedMaterial': proposed, 'expectedRewards': rewards,
                'actionValues': {'SORT': rewards['SORT'], 'INSPECT AGAIN': inspect},
                'selectedAction': action, 'value': value, 'transitions': transitions}
    return solve(belief['PLASTIC'], belief['PAPER'], remaining)

plan = solve_mdp(first['posteriors'], 2)
print('Immediate rewards:', plan['expectedRewards'])
print('Sequential action values:', plan['actionValues'])
for transition in plan['transitions']:
    print('Possible next observation:', transition)
print('Optimal action:', plan['selectedAction'])
assert isclose(plan['expectedRewards']['SORT'], 74/11)
assert isclose(plan['actionValues']['INSPECT AGAIN'], 14.374545454545455)
assert plan['selectedAction'] == 'INSPECT AGAIN'
assert solve_mdp(first['posteriors'], 0)['selectedAction'] == 'SORT'
assert solve_mdp(PRIORS, 0, wrong_reward=-40)['selectedAction'] == 'UNRESOLVED'

Immediate rewards: {'SORT': 6.727272727272727, 'INSPECT AGAIN': 2.818181818181818}
Sequential action values: {'SORT': 6.727272727272727, 'INSPECT AGAIN': 14.374545454545455}
Possible next observation: {'observation': 'PLASTIC', 'probability': 0.7727272727272727, 'posterior': {'PLASTIC': 0.9529411764705882, 'PAPER': 0.04705882352941177}, 'remainingInspections': 1, 'nextValue': 12.105882352941178, 'weightedValue': 9.354545454545455}
Possible next observation: {'observation': 'PAPER', 'probability': 0.2272727272727273, 'posterior': {'PLASTIC': 0.36, 'PAPER': 0.64}, 'remainingInspections': 1, 'nextValue': 9.687999999999999, 'weightedValue': 2.2018181818181817}
Optimal action: INSPECT AGAIN


## 3. Breadth-first search
The grid has four rows and four columns. Cell ID is $4r+c$ using zero-based coordinates. Robot starts at 12 = [3,0]. Plastic bin is 3 = [0,3]; Paper bin is 7 = [1,3]. The **non-target bin is blocked during navigation**, matching the controller.

BFS uses a FIFO queue and marks cells discovered when enqueued. Parent links reconstruct the path once the target is dequeued. Neighbors are processed **up, right, down, left**. `visitedCells` records dequeue order, not just the final route. Uniform edge costs make FIFO layers nondecreasing in distance, so the first target reached has a shortest path. Multiple shortest routes are possible; this neighbor order fixes a reproducible one. Complexity is $O(V+E)$ time and $O(V)$ memory. For unreachable targets, path is empty and path length is `None` (JSON `null`).

In [4]:
def bfs(start, target, obstacles=(), rows=4, columns=4):
    if type(rows) is not int or type(columns) is not int or rows < 1 or columns < 1:
        raise ValueError('Grid dimensions must be positive integers.')
    def valid(cell):
        return type(cell) is int and 0 <= cell < rows*columns
    if not valid(start) or not valid(target) or any(not valid(c) for c in obstacles):
        raise ValueError('Cell outside grid.')
    blocked = set(obstacles)
    if start in blocked or target in blocked:
        raise ValueError('Endpoints cannot be blocked.')
    queue, discovered, parents, visited = deque([start]), {start}, {}, []
    while queue:
        current = queue.popleft()
        visited.append(current)
        if current == target:
            path = [current]
            while path[-1] != start:
                path.append(parents[path[-1]])
            path.reverse()
            return {'path': path, 'pathLength': len(path)-1,
                    'visitedCells': visited, 'pathFound': True}
        row, column = divmod(current, columns)
        for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
            nr, nc = row+dr, column+dc
            if 0 <= nr < rows and 0 <= nc < columns:
                neighbor = nr*columns+nc
                if neighbor not in blocked and neighbor not in discovered:
                    discovered.add(neighbor)
                    parents[neighbor] = current
                    queue.append(neighbor)
    return {'path': [], 'pathLength': None, 'visitedCells': visited, 'pathFound': False}

route = bfs(12, 3, [7])
print('Route:', route)
assert route['path'] == [12, 8, 4, 0, 1, 2, 3]
assert bfs(0, 3, [1, 7])['path'] == [0, 4, 5, 6, 2, 3]
assert not bfs(12, 3, [2, 7])['pathFound']
assert bfs(3, 3)['pathLength'] == 0

Route: {'path': [12, 8, 4, 0, 1, 2, 3], 'pathLength': 6, 'visitedCells': [12, 8, 13, 4, 9, 14, 0, 5, 10, 15, 1, 6, 11, 2, 3], 'pathFound': True}


## 4. Complete agent: Object → Sensor → Bayes → MDP → BFS → Result
The sensor reproduces the application's unsigned 32-bit linear congruential generator exactly: `next = (1664525*seed + 1013904223) mod 2**32`. The draw selects PLASTIC with the configured probability for the true material; otherwise PAPER. Predefined observations bypass sampling for controlled demonstrations.

The loop consumes at most the configured number of reads. The first read precedes planning, so the planner receives `maxInspections - reads_used`. Only SORT triggers BFS. An exhausted script or unresolved terminal belief stops safely; blocked destinations return unreachable. The final correctness check compares the selected material with ground truth **after** the policy acts.

In [5]:
def sample_sensor(material, seed, model):
    if material not in MATERIALS or type(seed) is not int or not 0 <= seed <= 0xffffffff:
        raise ValueError('Invalid material or 32-bit seed.')
    next_seed = (1664525*seed + 1013904223) & 0xffffffff
    draw = next_seed / 4294967296
    return ('PLASTIC' if draw < model['PLASTIC'][material] else 'PAPER'), next_seed, draw

def simulate(config):
    params = config['parameters']
    model = sensor_model(params)
    belief = dict(config['initialPriors'])
    validate_belief(belief)
    material = OBJECTS[config['objectId']]
    start, obstacles, source = config['start'], config['obstacles'], config['sensor']
    if start in BINS.values() or start in obstacles or any(c in BINS.values() for c in obstacles):
        raise ValueError('Robot and bins must be preserved.')
    inspections, destination, route = [], None, None
    outcome, predicted, moves = 'unresolved', None, 0
    seed = source.get('seed', 0)
    for index in range(params['maxInspections']):
        if source['mode'] == 'predefined':
            if index >= len(source['observations']):
                break
            observation = source['observations'][index]
        elif source['mode'] == 'seeded':
            observation, seed, _ = sample_sensor(material, seed, model)
        else:
            raise ValueError('Invalid observation source.')
        inference = bayes(observation, belief, model)
        belief = inference['posteriors']  # carry posterior forward
        decision = solve_mdp(belief, params['maxInspections']-index-1, model, -params['incorrectSortPenalty'])
        inspections.append({'observation': observation, 'bayesian': inference, 'decision': decision})
        predicted = decision['proposedMaterial']
        if decision['selectedAction'] == 'UNRESOLVED':
            break
        if decision['selectedAction'] == 'SORT':
            destination = BINS[predicted]
            blocked = list(obstacles) + [cell for cell in BINS.values() if cell != destination]
            route = bfs(start, destination, blocked)
            if route['pathFound']:
                moves = route['pathLength']
                outcome = 'correct' if predicted == material else 'incorrect'
            else:
                outcome = 'unreachable'
            break
    return {'inspections': inspections, 'posterior': belief, 'destination': destination, 'bfs': route,
            'result': {'outcome': outcome, 'trueMaterial': material, 'predictedMaterial': predicted,
                       'destination': destination, 'inspections': len(inspections), 'moves': moves}}

sample_input = {'objectId': 'plastic-bottle', 'start': 12, 'obstacles': [],
                'sensor': {'mode': 'seeded', 'seed': 42},
                'initialPriors': dict(PRIORS), 'parameters': dict(DEFAULT_PARAMETERS)}
example = simulate(sample_input)
print('Sample input:', sample_input)
for number, inspection in enumerate(example['inspections'], 1):
    b, d = inspection['bayesian'], inspection['decision']
    print(f"Inspection {number}: {inspection['observation']}; evidence={b['evidence']:.8f}; "
          f"Plastic={b['posteriors']['PLASTIC']:.8%}; Paper={b['posteriors']['PAPER']:.8%}")
    print('  h=', d['horizon'], 'Q=', d['actionValues'], 'action=', d['selectedAction'])
print('BFS:', example['bfs'])
print('Final result:', example['result'])
assert example['result']['outcome'] == 'correct'
assert example['result']['inspections'] == 3

Sample input: {'objectId': 'plastic-bottle', 'start': 12, 'obstacles': [], 'sensor': {'mode': 'seeded', 'seed': 42}, 'initialPriors': {'PLASTIC': 0.5, 'PAPER': 0.5}, 'parameters': {'sensorAccuracy': 0.85, 'incorrectSortPenalty': 8, 'maxInspections': 3}}
Inspection 1: PLASTIC; evidence=0.55000000; Plastic=81.81818182%; Paper=18.18181818%
  h= 2 Q= {'SORT': 6.727272727272727, 'INSPECT AGAIN': 14.374545454545455} action= INSPECT AGAIN
Inspection 2: PLASTIC; evidence=0.77272727; Plastic=95.29411765%; Paper=4.70588235%
  h= 1 Q= {'SORT': 9.152941176470588, 'INSPECT AGAIN': 12.105882352941178} action= INSPECT AGAIN
Inspection 3: PLASTIC; evidence=0.86705882; Plastic=98.91451832%; Paper=1.08548168%
  h= 0 Q= {'SORT': 9.804613297150611, 'INSPECT AGAIN': None} action= SORT
BFS: {'path': [12, 8, 4, 0, 1, 2, 3], 'pathLength': 6, 'visitedCells': [12, 8, 13, 4, 9, 14, 0, 5, 10, 15, 1, 6, 11, 2, 3], 'pathFound': True}
Final result: {'outcome': 'correct', 'trueMaterial': 'PLASTIC', 'predictedMaterial

## 5. Cross-language regression checks and edge cases
The following assertions compare independently computed Python results with saved results generated by the **actual TypeScript controller**, including all observation branches of each chosen Bellman root, posterior updates, path and dequeue order, and final outcome. Numeric comparison tolerates roundoff only. Seven fixtures cover all three defense demonstrations plus Paper, an incorrect observation leading to wrong sorting, unreachable navigation, and unresolved classification.

Regenerate references from the repository root with `node --experimental-strip-types scripts/submission-fixtures.mjs`; verify unchanged references with `--check`. Do not edit fixture outputs to force a passing comparison.

In [6]:
notebook_dir = globals().get('NOTEBOOK_DIR', Path.cwd())
reference_file = notebook_dir / 'reference_results.json'
if not reference_file.exists():
    reference_file = Path.cwd() / 'notebooks' / 'reference_results.json'
references = json.loads(reference_file.read_text(encoding='utf-8'))['scenarios']

def equivalent(actual, expected):
    if isinstance(expected, dict):
        return isinstance(actual, dict) and actual.keys() == expected.keys() and all(equivalent(actual[k], v) for k, v in expected.items())
    if isinstance(expected, list):
        return isinstance(actual, list) and len(actual) == len(expected) and all(equivalent(a, b) for a, b in zip(actual, expected))
    if type(expected) in (float, int):
        return type(actual) in (float, int) and isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-9)
    return type(actual) is type(expected) and actual == expected

for fixture in references:
    actual = simulate(fixture['config'])
    expected = {k: fixture[k] for k in actual}
    expected['result'] = {k: fixture['result'][k] for k in actual['result']}
    assert equivalent(actual, expected), f"Cross-language mismatch: {fixture['id']}"
    print(f"PASS {fixture['id']}: {actual['result']['outcome']}, {actual['result']['inspections']} reads, {actual['result']['moves']} moves")

def rejects(function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except (ValueError, TypeError, KeyError):
        return
    raise AssertionError('Invalid input was accepted')

rejects(bayes, 'METAL')
rejects(bayes, 'PLASTIC', {'PLASTIC': 0.9, 'PAPER': 0.9})
rejects(bayes, 'PLASTIC', {'PLASTIC': float('nan'), 'PAPER': 0.5})
rejects(bayes, 'PLASTIC', PRIORS, {'PLASTIC': {'PLASTIC': 0, 'PAPER': 0}, 'PAPER': {'PLASTIC': 1, 'PAPER': 1}})
rejects(solve_mdp, PRIORS, -1)
rejects(solve_mdp, PRIORS, 11)
rejects(sample_sensor, 'PLASTIC', -1, MODEL)
rejects(bfs, 12, 3, [3])
rejects(bfs, 16, 3)
rejects(sensor_model, {**DEFAULT_PARAMETERS, 'sensorAccuracy': 1.1})
for observations in [('PLASTIC',)*3, ('PAPER',)*3, ('PLASTIC','PAPER','PAPER')]:
    belief = dict(PRIORS)
    for observation in observations:
        belief = bayes(observation, belief)['posteriors']
        assert isclose(sum(belief.values()), 1)
    decision = solve_mdp(belief, 2)
    assert isclose(sum(t['probability'] for t in decision['transitions']), 1)
# Impossible observation branches must be skipped, not divided by zero.
perfect = {'PLASTIC': {'PLASTIC': 1, 'PAPER': 0}, 'PAPER': {'PLASTIC': 0, 'PAPER': 1}}
assert len(solve_mdp({'PLASTIC': 1, 'PAPER': 0}, 2, perfect)['transitions']) == 1
print('PASS: invalid inputs, normalization, terminal policies, impossible branches, and BFS edge cases.')
print('All notebook assertions passed.')

PASS A: correct, 3 reads, 6 moves
PASS B: correct, 3 reads, 5 moves
PASS C: correct, 1 reads, 5 moves
PASS paper: correct, 1 reads, 5 moves
PASS incorrect: incorrect, 1 reads, 6 moves
PASS unreachable: unreachable, 1 reads, 0 moves
PASS unresolved: unresolved, 3 reads, 0 moves
PASS: invalid inputs, normalization, terminal policies, impossible branches, and BFS edge cases.
All notebook assertions passed.


## Conclusions and limitations
Bayes manages uncertain perception, Bellman planning selects a bounded sensing/sorting policy, and BFS executes navigation on an unweighted floor. Seeded and scripted runs are reproducible. A confident classification can still be wrong: confidence is conditional on the assumed model, not a guarantee.

Two material classes, a known stationary sensor, conditionally independent readings, a 4x4 static floor, deterministic movement, and hand-designed rewards simplify the real problem. Positive inspection rewards encourage additional reads. This is planning with a known model, not training or reinforcement learning. The notebook reproduces algorithmic decisions and outcomes; it does not reproduce browser animation scheduling, localStorage, replay controls, or accessibility behavior. Those are covered by the website tests.